# Problema 8.4 -- Localizzazione di hub con costo di connessione massimo

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/modellazione-mip/blob/main/notebooks/fam08_4_hub.ipynb)

Due link: attivazione (aggregata, come nello scheduling 7.2) e variabile di
massimo z_j = max_i {c_ij : x_ij = 1} (stesso schema del ritardo 7.7). L'euristica
next-fit è quella generica di euristiche.py: gli hub sono le "macchine" (capacità
k) e i terminali i "lavori" (tempo unitario, indipendente dalla macchina).

Il capitolo completo — modello, dati, risultati e analisi di sensitività — è [sul sito](https://fabiofurini.github.io/modellazione-mip/localizzazione-4/).

## Preparazione

La cella qui sotto installa `gurobipy` e scarica i tre moduli comuni del corso:
`stile.py` (palette), `mip.py` (rilassamento, duale, bound) ed `euristiche.py`
(next-fit, first-fit, best-fit). La licenza inclusa nel pacchetto pip è limitata a **2000
variabili e 2000 vincoli**: le istanze del corso sono piccole e ci stanno tutte
con ampio margine. Per istanze più grandi si attiva la licenza accademica
gratuita da [portal.gurobi.com](https://portal.gurobi.com).

In [ ]:
# Ambiente: il solver e i moduli comuni del corso.
# In locale usa il python/stile.py del repository; su Colab installa e scarica quello che manca.
import importlib.util
import subprocess
import sys
import urllib.request
from pathlib import Path

if importlib.util.find_spec("gurobipy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gurobipy", "matplotlib", "pandas", "scipy"], check=True)

for modulo in ('stile', 'mip', 'euristiche'):                     # stile grafico e utilità del corso
    if importlib.util.find_spec(modulo) is None:
        locale = next((p for p in (Path(f"../python/{modulo}.py"), Path(f"python/{modulo}.py"))
                       if p.exists()), None)
        if locale is not None:
            sys.path.insert(0, str(locale.parent.resolve()))   # notebook aperto nel repository
        else:
            urllib.request.urlretrieve(f"https://raw.githubusercontent.com/fabiofurini/modellazione-mip/main/python/{modulo}.py", f"{modulo}.py")   # Colab

In [ ]:
import gurobipy as gp
import pandas as pd
from gurobipy import GRB

from euristiche import matrice, next_fit
from mip import (due_rilassamenti, frazione, nuovo_modello, registra_bound,
                 rilassamento, risolvi, stampa_soluzione, valuta)
from stile import intestazione, plt, salva_dati, salva_figura

R = range

# ---------- 1. MODELLO E ISTANZA ----------

intestazione("4. Localizzazione di hub: attivazione e costo di connessione massimo")
c4 = [[5, 10, 2], [5, 4, 6], [5, 4, 6]]   # costo di connessione terminale i -> hub j
f4 = [5, 6, 7]                             # costo di attivazione hub j
k4 = 2                                     # capacità di ciascun hub
n, m = 3, 3
salva_dati(pd.DataFrame([{"terminale": i + 1, "hub": j + 1, "c": c4[i][j]}
                         for i in R(n) for j in R(m)]), "hub4_costi")
salva_dati(pd.DataFrame({"hub": R(1, m + 1), "f": f4}), "hub4_attivazione")


def modello_4(c, f, k):
    n, m = len(c), len(f)
    mod = nuovo_modello("hub_max")
    x = mod.addVars(n, m, vtype=GRB.BINARY, name="x")
    y = mod.addVars(m, vtype=GRB.BINARY, name="y")
    z = mod.addVars(m, name="z")
    mod.setObjective(gp.quicksum(f[j] * y[j] for j in R(m)) + z.sum(), GRB.MINIMIZE)
    mod.addConstrs((gp.quicksum(x[i, j] for j in R(m)) == 1 for i in R(n)), name="assegnamento")
    mod.addConstrs((-gp.quicksum(x[i, j] for i in R(n)) + k * y[j] >= 0 for j in R(m)), name="attivazione")
    mod.addConstrs((-c[i][j] * x[i, j] + z[j] >= 0 for i in R(n) for j in R(m)), name="massimo")
    return mod, x, y, z


def duale_4(c, f, k):
    """max sum_i alpha_i;  alpha_i - beta_j - c_ij gamma_ij <= 0;  k beta_j <= f_j;
    sum_i gamma_ij <= 1;  alpha libero, beta,gamma >= 0."""
    n, m = len(c), len(f)
    dl = nuovo_modello("duale_hub")
    alpha = dl.addVars(n, lb=-GRB.INFINITY, name="alpha")
    beta = dl.addVars(m, name="beta")
    gamma = dl.addVars(n, m, name="gamma")
    dl.setObjective(alpha.sum(), GRB.MAXIMIZE)
    dl.addConstrs((alpha[i] - beta[j] - c[i][j] * gamma[i, j] <= 0 for i in R(n) for j in R(m)), name="rc_x")
    dl.addConstrs((k * beta[j] <= f[j] for j in R(m)), name="rc_y")
    dl.addConstrs((gp.quicksum(gamma[i, j] for i in R(n)) <= 1 for j in R(m)), name="rc_z")
    return dl


m4, x4, y4, z4 = modello_4(c4, f4, k4)

# ---------- 2. EURISTICA COSTRUTTIVA (UPPER BOUND) ----------

print("Euristica next-fit: si riempiono gli hub uno alla volta fino a k terminali,")
print("poi si passa al successivo (stessa euristica generica dei problemi di scheduling).")
t4 = matrice([1] * n, m)   # tempo unitario per ogni terminale, indipendente dall'hub
a4 = [k4] * m                    # capacità residua di ciascun hub
esito4 = next_fit(t4, a4)
esito4.traccia.stampa()
assert esito4.ok
ye = esito4.y
ze = [0.0] * m
for j in R(m):
    if ye[j]:
        ze[j] = max(c4[i][j] for i in R(n) if esito4.x.get((i, j)) == 1)
ub4 = sum(f4[j] * ye[j] for j in R(m)) + sum(ze)
print(f"  y = {ye}, z = {ze}  ->  ub = {frazione(ub4)}")

# ---------- 3. RILASSAMENTO LP E DUALE (LOWER BOUND) ----------

d4 = duale_4(c4, f4, k4)
beta_mano = [f4[j] / k4 for j in R(m)]     # il massimo ammesso da k*beta_j <= f_j
alpha_mano = min(beta_mano)                # deve reggere per OGNI hub j, non solo il più conveniente
mano = {f"gamma[{i},{j}]": 0.0 for i in R(n) for j in R(m)}
mano.update({f"beta[{j}]": beta_mano[j] for j in R(m)})
mano.update({f"alpha[{i}]": alpha_mano for i in R(n)})
lb4, viol = valuta(d4, mano)
assert viol <= 1e-9, viol
print(f"Soluzione duale a mano: gamma = 0, beta_j = f_j/k = {[frazione(b) for b in beta_mano]}, "
      f"alpha_i = min_j beta_j = {frazione(alpha_mano)}  ->  lb = {frazione(lb4)}")
zlp4, zlp4r, _ = due_rilassamenti(m4, d4)

# ---------- 4. SOLUZIONE OTTIMA DEL MILP ----------

z4v = risolvi(m4)
print("Soluzione ottima del MILP:")
stampa_soluzione(m4, solo_non_nulle=True)
riga = registra_bound("4 hub", ub4, lb4, zlp4, zlp4r, z4v, senso="min")
salva_dati(pd.DataFrame([riga]), "hub4_bound")

# ---------- 5. DOMANDE DI MODELLAZIONE AGGIUNTIVE ----------

varianti = {}


def variante(nome, mod):
    z = risolvi(mod)
    print(f"  {nome:70s} z = {frazione(z)}")
    return z


# 4a: si AGGIUNGONO i link disaggregati x_ij <= y_j al vincolo aggregato
mod, x, y, z = modello_4(c4, f4, k4)
mod.addConstrs((x[i, j] <= y[j] for i in R(n) for j in R(m)), name="attivazione_disaggregata")
varianti["4a"] = variante("4a. Link disaggregati AGGIUNTI a quello aggregato (x_ij <= y_j)", mod)
zlp_4a, _, _ = rilassamento(mod, rafforzato=True)
zlp_base, _, _ = rilassamento(modello_4(c4, f4, k4)[0], rafforzato=True)
print(f"      rilassamento: z(LP+) passa da {frazione(zlp_base)} a {frazione(zlp_4a)}: i link")
print("      disaggregati sono disuguaglianze valide implicate dal vincolo aggregato sui")
print("      punti interi, ma non dal rilassamento, e lo rafforzano.")

# 4a-bis: il tranello. Se si SOSTITUISCE il vincolo aggregato con i soli link
# disaggregati, si perde anche la capacita' k: il modello non e' piu' quello del
# problema. Va tenuta esplicitamente, oppure si parla di aggiunta e non di sostituzione.
mod, x, y, z = modello_4(c4, f4, k4)
mod.update()
mod.remove([cc for cc in mod.getConstrs() if cc.ConstrName.startswith("attivazione")])
mod.update()
mod.addConstrs((x[i, j] <= y[j] for i in R(n) for j in R(m)), name="solo_disaggregato")
varianti["4a_senza_capacita"] = variante(
    "4a'. SOSTITUENDO l'aggregato con i soli link disaggregati (capacita' persa)", mod)
mod, x, y, z = modello_4(c4, f4, k4)
mod.update()
mod.remove([cc for cc in mod.getConstrs() if cc.ConstrName.startswith("attivazione")])
mod.update()
mod.addConstrs((x[i, j] <= y[j] for i in R(n) for j in R(m)), name="solo_disaggregato")
mod.addConstrs((gp.quicksum(x[i, j] for i in R(n)) <= k4 for j in R(m)), name="capacita")
varianti["4a_con_capacita"] = variante(
    "4a''. Sostituzione corretta: link disaggregati + capacita' separata", mod)
assert varianti["4a_senza_capacita"] < varianti["4a"], "senza capacita' l'ottimo scende"
assert varianti["4a_con_capacita"] == varianti["4a"], "con la capacita' l'ottimo non cambia"
# 4b: il terminale 1 non può essere connesso all'hub 2
mod, x, y, z = modello_4(c4, f4, k4)
mod.addConstr(x[0, 1] == 0, name="terminale1_non_hub2")
varianti["4b"] = variante("4b. Il terminale 1 non può connettersi all'hub 2 (x_12 = 0)", mod)
salva_dati(pd.DataFrame({"variante": list(varianti), "z": list(varianti.values())}), "hub4_varianti")

# ---------- 6. FIGURE ----------

fig, ax = plt.subplots(figsize=(6.4, 3.2))
colori = ["#16324A", "#0E7490", "#CA6F1E"]
for j in R(m):
    if y4[j].X > 0.5:
        assegnati = [i + 1 for i in R(n) if x4[i, j].X > 0.5]
        ax.barh(j, z4[j].X, color=colori[j % 3], label=f"hub {j + 1}: terminali {assegnati}")
ax.set_yticks(R(m))
ax.set_yticklabels([f"hub {j + 1}" for j in R(m)])
ax.set_xlabel("costo di connessione massimo $z_j$")
ax.set_title(f"Soluzione ottima (z = {frazione(z4v)})")
ax.legend(fontsize=7, loc="lower right")
salva_figura(fig, "cap08_hub_ottimo")
print("Fine.")

---

Notebook generato da `python/fam08_4_hub.py` con `python3 python/genera_notebook.py`:
le modifiche si fanno sullo script, non qui.

Materiale didattico di [Fabio Furini](https://sites.google.com/view/fabiofurini/home-page) — DIAG, Sapienza Università di Roma.
Testi, figure e dati [CC BY 4.0](https://github.com/fabiofurini/modellazione-mip/blob/main/LICENSE),
codice [MIT](https://github.com/fabiofurini/modellazione-mip/blob/main/LICENSE-CODE).